In [4]:
import warnings
warnings.filterwarnings("ignore")

# LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()
key = os.getenv("GROQ_API_KEY")
chat_model = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=key
)

print(f"GROQ API key found: {bool(key)}")

# Pydantic
from pydantic import BaseModel


class GoalDefine(BaseModel):
    skills: list[str]
    roadmap: list[str]
    first_project: str


structured_llm = chat_model.with_structured_output(GoalDefine)

# LangGraph
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import Annotated, TypedDict


class StudentState(TypedDict):

    goal: str
    skills: list[str]
    roadmap: list[str]
    first_project: str
    messages: Annotated[list, add_messages]

# Goal Analysis Node
def analysis_goal(state: StudentState):

    goal = state["goal"]

    print(f"Goal: {goal}")
    prompt = f"""
Analyze this student's goal:

{goal}

Identify:

1. Required skills
2. Learning roadmap
3. First practical project

Make the roadmap practical and ordered.
"""
    response = structured_llm.invoke(prompt)

    return {
        "skills": response.skills,

        "roadmap": response.roadmap,

        "first_project": response.first_project,

        "messages": [
            (
                "assistant",
                f"Goal analysis completed for: {goal}"
            )
        ]
    }

def project_planer(state:StudentState):
    project = state['first_project']
    prompt = f""" 
    Create a step-by-step implementation plan for this project:

{project}

Include:
1. Dataset
2. Data preprocessing
3. Model
4. Training
5. Evaluation
6. Deployment
"""

    response = chat_model.invoke(prompt) 
    return {
        "messages": [response]
    }
# Build Graph

graph = StateGraph(StudentState)

graph.add_node("analysis",analysis_goal)
graph.add_node("planer",project_planer)

graph.add_edge(START,"analysis")
graph.add_edge("analysis", "planer")
graph.add_edge("planer",END)

# Compile
app = graph.compile()

# Test
result = app.invoke({

    "goal": "I want to become an ML engineer",

    "skills": [],

    "roadmap": [],

    "first_project": "",

    "messages": [
        (
            "human",
            "I want to become an ML engineer"
        )
    ]
})


print("Skills:", result["skills"])
print("Roadmap:", result["roadmap"])
print("First Project:", result["first_project"])

GROQ API key found: True
Goal: I want to become an ML engineer
Skills: ['Mathematics: Linear algebra, calculus, probability & statistics', 'Programming: Python (core), Git, Linux command line', 'Data manipulation: NumPy, pandas', 'Visualization: Matplotlib, Seaborn, Plotly', 'Machine learning: Scikit‑learn, model evaluation, feature engineering', 'Deep learning frameworks: TensorFlow, PyTorch', 'Neural network architectures: CNN, RNN, Transformers basics', 'Model optimization: hyper‑parameter tuning, regularization techniques', 'Software engineering: modular code, testing (pytest), logging', 'MLOps tools: Docker, CI/CD (GitHub Actions), experiment tracking (MLflow)', 'Data pipelines: Airflow/Prefect basics, ETL concepts', 'Cloud platforms: AWS/GCP/Azure ML services, serverless inference', 'Communication: technical writing, presenting results, documentation']
Roadmap: ['Foundations: Linear Algebra, Calculus, Probability & Statistics', 'Python programming: data structures, OOP, virtual e